# Punch Annotator

A lightweight Jupyter notebook for annotating punch contact frames in boxing/MMA video.

## What this does

- Loads a video and displays it one frame at a time
- Lets you scrub through frames with arrow keys
- Hotkey a punch type to log the current frame
- Saves annotations to a CSV that accumulates across videos

## Output schema

Each row in `annotations.csv`:

| video_id | frame_index | punch_type | occluded | notes | timestamp |
|---|---|---|---|---|---|

## First-time setup

Run the install cell once, then skip it on subsequent sessions.

## 1. Install dependencies (run once)

In [7]:
# Run once per environment
!pip install av ipywidgets ipyevents pandas Pillow numpy

## 2. Imports

In [8]:
from pathlib import Path
from datetime import datetime
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image

import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display
from ipyevents import Event

import av

## 3. Configuration

Edit `VIDEO_PATH` for each new video. `CSV_PATH` stays the same across the whole project.

In [13]:
# === EDIT THESE ===

VIDEO_PATH   = "../data/downloaded-videos/V9.mp4"   # path to the video you're annotating now
CSV_PATH     = "../data/annotations.csv"            # shared CSV across all videos
SKELETON_DIR = "../data/raw/Skeleton_data"          # folder containing corresponding .npy files

# Hotkey -> class label
CLASSES = {
    's': 'straight',
    ' ': 'straight',
    'h': 'hook',
    'a': 'hook',
    'u': 'uppercut',
    'd': 'uppercut',
    'n': 'none',
    'e': 'none',
}

# Auto-save after this many new annotations
AUTOSAVE_EVERY = 10

# Display width for the video frame (pixels)
DISPLAY_WIDTH = 800

In [14]:
import os
video_dir = "../data/downloaded-videos"
skel_dir  = "../data/raw/Skeleton_data"

videos = sorted([f for f in os.listdir(video_dir) if f.endswith('.mp4')])
print(f"Found {len(videos)} videos in {video_dir}:")
for v in videos:
    skel_file = v.replace('.mp4', '.npy')
    skel_exists = os.path.exists(os.path.join(skel_dir, skel_file))
    status = "[SKEL FOUND]" if skel_exists else "[NO SKEL]"
    print(f" - {os.path.join(video_dir, v)} {status}")

Found 12 videos in ../data/downloaded-videos:
 - ../data/downloaded-videos/V1.mp4 [NO SKEL]
 - ../data/downloaded-videos/V10.mp4 [NO SKEL]
 - ../data/downloaded-videos/V11.mp4 [NO SKEL]
 - ../data/downloaded-videos/V12.mp4 [NO SKEL]
 - ../data/downloaded-videos/V2.mp4 [NO SKEL]
 - ../data/downloaded-videos/V3.mp4 [NO SKEL]
 - ../data/downloaded-videos/V4.mp4 [NO SKEL]
 - ../data/downloaded-videos/V5.mp4 [NO SKEL]
 - ../data/downloaded-videos/V6.mp4 [NO SKEL]
 - ../data/downloaded-videos/V7.mp4 [NO SKEL]
 - ../data/downloaded-videos/V8.mp4 [NO SKEL]
 - ../data/downloaded-videos/V9.mp4 [NO SKEL]


## 4. The Annotator class (with PyAV backend)

Run this cell once. You don't need to edit it.

In [15]:
class PyAVReader:
    """Frame-accurate random-access video reader built on PyAV.

    Strategy:
      - Walk the container once at open time to build a list of keyframe PTS
        and estimate total frame count + fps.
      - To seek to frame N: seek to the nearest keyframe <= N, then decode
        forward until we land on N.
      - Cache the most recently decoded frames so scrubbing +/- 1 is instant.
    """

    def __init__(self, path, cache_size=64):
        self.path = str(path)
        self.container = av.open(self.path)
        self.stream = self.container.streams.video[0]
        self.stream.thread_type = "AUTO"

        # fps
        if self.stream.average_rate is not None:
            self.fps = float(self.stream.average_rate)
        else:
            self.fps = float(self.stream.guessed_rate or 30.0)

        # Total frame count: trust metadata if present, else count.
        self.total_frames = self.stream.frames
        if not self.total_frames:
            # Expensive fallback: count by walking once. OK for <30min videos.
            self.total_frames = sum(1 for _ in self.container.decode(video=0))
            # Reopen since the container is now exhausted
            self.container.close()
            self.container = av.open(self.path)
            self.stream = self.container.streams.video[0]
            self.stream.thread_type = "AUTO"

        self.time_base = self.stream.time_base
        self.cache = {}           # frame_index -> numpy array
        self.cache_order = []     # LRU order
        self.cache_size = cache_size
        self._last_decoded_idx = -1
        self._decode_iter = None  # ongoing forward decode, if any

    def _pts_for_frame(self, frame_index):
        # frame -> seconds -> PTS in stream time_base
        seconds = frame_index / self.fps
        return int(seconds / float(self.time_base))

    def _cache_put(self, idx, arr):
        if idx in self.cache:
            self.cache_order.remove(idx)
        elif len(self.cache_order) >= self.cache_size:
            drop = self.cache_order.pop(0)
            self.cache.pop(drop, None)
        self.cache[idx] = arr
        self.cache_order.append(idx)

    def get_frame(self, frame_index):
        frame_index = max(0, min(int(frame_index), self.total_frames - 1))
        if frame_index in self.cache:
            return self.cache[frame_index]

        # Fast path: next frame in sequence, continue the existing decode iterator
        if frame_index == self._last_decoded_idx + 1 and self._decode_iter is not None:
            try:
                frame = next(self._decode_iter)
                arr = frame.to_ndarray(format="rgb24")
                self._last_decoded_idx = frame_index
                self._cache_put(frame_index, arr)
                return arr
            except StopIteration:
                self._decode_iter = None

        # Seek path: jump to keyframe at or before target PTS, decode forward
        target_pts = self._pts_for_frame(frame_index)
        self.container.seek(target_pts, any_frame=False, backward=True, stream=self.stream)
        self._decode_iter = self.container.decode(video=0)

        last_arr = None
        current_idx = -1
        for frame in self._decode_iter:
            # Map this frame's PTS back to a frame index
            if frame.pts is None:
                continue
            t = float(frame.pts * self.time_base)
            current_idx = int(round(t * self.fps))
            if current_idx > frame_index:
                break
            last_arr = frame.to_ndarray(format="rgb24")
            if current_idx == frame_index:
                self._last_decoded_idx = frame_index
                self._cache_put(frame_index, last_arr)
                return last_arr

        # Fell off the end -- return the last decoded frame we saw
        if last_arr is not None:
            self._last_decoded_idx = current_idx
            self._cache_put(current_idx, last_arr)
            return last_arr
        raise RuntimeError(f"Could not decode frame {frame_index}")

    def __len__(self):
        return self.total_frames

    def close(self):
        self.container.close()


class Annotator:
    def __init__(self, video_path, csv_path):
        self.video_path = Path(video_path)
        self.video_id = self.video_path.name
        self.csv_path = Path(csv_path)

        if not self.video_path.exists():
            raise FileNotFoundError(f"Video not found: {self.video_path}")

        # Make sure the CSV's parent directory exists
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)

        self.reader = PyAVReader(self.video_path)
        self.total_frames = len(self.reader)
        self.fps = self.reader.fps
        self.current_frame = 0

        self.annotations = self._load_existing()
        self.occluded_flag = False
        self.pending_changes = 0
        self.last_action_msg = "Ready. Use keyboard shortcuts (ensure UI is focused)."

        # Resume: jump to just after the last labeled frame for this video
        video_frames = [a["frame_index"] for a in self.annotations if a["video_id"] == self.video_id]
        if video_frames:
            self.current_frame = min(max(video_frames) + 1, self.total_frames - 1)

        self._build_ui()
        self._render()

    # --- persistence ---

    def _load_existing(self):
        if self.csv_path.exists():
            df = pd.read_csv(self.csv_path)
            return df.to_dict("records")
        return []

    def _save_csv(self):
        df = pd.DataFrame(self.annotations)
        tmp = self.csv_path.with_suffix(self.csv_path.suffix + ".tmp")
        df.to_csv(tmp, index=False)
        tmp.replace(self.csv_path)
        self.pending_changes = 0

    # --- UI ---

    def _build_ui(self):
        self.status = widgets.HTML()
        self.img_widget = widgets.Image(format="jpeg", width=DISPLAY_WIDTH)
        
        # Use a Output widget to catch potential errors
        self.output = widgets.Output()

        def mk(desc, on_click, width="80px"):
            b = widgets.Button(description=desc, layout=Layout(width=width))
            b.on_click(on_click)
            return b

        nav = widgets.HBox([
            mk("<<  Shift+Left", lambda _: self._step(-10), "130px"),
            mk("< Left",         lambda _: self._step(-1),  "90px"),
            mk("Right >",        lambda _: self._step(+1),  "90px"),
            mk("Shift+Right  >>",lambda _: self._step(+10), "140px"),
        ])

        label_buttons = [
            mk(f"{cls} ({key})", lambda _, c=cls: self._annotate(c), "100px")
            for key, cls in CLASSES.items()
        ]
        labels = widgets.HBox(label_buttons)

        actions = widgets.HBox([
            mk("occluded (o)", lambda _: self._toggle_occluded(), "110px"),
            mk("undo (z)",     lambda _: self._undo(),             "90px"),
            mk("save (s)",     lambda _: self._manual_save(),      "90px"),
        ])

        self.jump_input = widgets.IntText(value=0, description="Frame:", layout=Layout(width="180px"))
        jump_btn = mk("Go", lambda _: self._seek(self.jump_input.value), "50px")
        jump = widgets.HBox([self.jump_input, jump_btn])

        self.container = widgets.VBox([
            self.status, self.img_widget, nav, labels, actions, jump, self.output
        ])

        # Listen to events on the whole container VBox
        self.evt = Event(
            source=self.container,
            watched_events=["keydown"],
            prevent_default_action=True,
        )
        self.evt.on_dom_event(self._on_key)

    def _on_key(self, event):
        with self.output:
            key = event.get("key", "")
            shift = event.get("shiftKey", False)

            if key == "ArrowLeft":
                self._step(-10 if shift else -1)
            elif key == "ArrowRight":
                self._step(+10 if shift else +1)
            elif key in CLASSES:
                self._annotate(CLASSES[key])
            elif key == "o":
                self._toggle_occluded()
            elif key == "z":
                self._undo()
            elif key == "s":
                self._manual_save()

    # --- actions ---

    def _step(self, delta):
        self._seek(self.current_frame + delta)

    def _seek(self, frame):
        self.current_frame = int(max(0, min(frame, self.total_frames - 1)))
        self._render()

    def _annotate(self, cls):
        self.annotations.append({
            "video_id":    self.video_id,
            "frame_index": self.current_frame,
            "punch_type":  cls,
            "occluded":    self.occluded_flag,
            "notes":       "",
            "timestamp":   datetime.now().isoformat(timespec="seconds"),
        })
        self.pending_changes += 1
        msg = f"labeled frame {self.current_frame} as {cls}"
        if self.occluded_flag:
            msg += " [occluded]"
        self.occluded_flag = False

        if self.pending_changes >= AUTOSAVE_EVERY:
            self._save_csv()
            msg += f" -- auto-saved, {len(self.annotations)} total"

        self.last_action_msg = msg
        self._render_status()

    def _toggle_occluded(self):
        self.occluded_flag = not self.occluded_flag
        self.last_action_msg = f"occluded flag for next label: {self.occluded_flag}"
        self._render_status()

    def _undo(self):
        for i in range(len(self.annotations) - 1, -1, -1):
            if self.annotations[i]["video_id"] == self.video_id:
                removed = self.annotations.pop(i)
                self._save_csv()
                self.last_action_msg = (
                    f"undid {removed['punch_type']} at frame {removed['frame_index']}"
                )
                self._seek(removed["frame_index"])
                return
        self.last_action_msg = "nothing to undo for this video"
        self._render_status()

    def _manual_save(self):
        self._save_csv()
        self.last_action_msg = f"saved {len(self.annotations)} annotations"
        self._render_status()

    # --- render ---

    def _render(self):
        frame = self.reader.get_frame(self.current_frame)  # H x W x 3, RGB
        img = Image.fromarray(frame)
        if img.width > DISPLAY_WIDTH:
            ratio = DISPLAY_WIDTH / img.width
            img = img.resize((DISPLAY_WIDTH, int(img.height * ratio)))
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=85)
        self.img_widget.value = buf.getvalue()
        self._render_status()

    def _render_status(self):
        video_count = sum(1 for a in self.annotations if a["video_id"] == self.video_id)
        time_str = f"{self.current_frame / self.fps:.2f}s" if self.fps > 0 else "?"
        occl_tag = " <span style='color:#c00'><b>[OCCLUDED NEXT]</b></span>" if self.occluded_flag else ""
        unsaved = f" <span style='color:#888'>({self.pending_changes} unsaved)</span>" if self.pending_changes else ""
        self.status.value = (
            f"<b>{self.video_id}</b> | "
            f"frame <b>{self.current_frame}</b> / {self.total_frames - 1} "
            f"({time_str} @ {self.fps:.1f} fps) | "
            f"this video: <b>{video_count}</b> | total: <b>{len(self.annotations)}</b>"
            f"{unsaved}{occl_tag}"
            f"<br><i>{self.last_action_msg}</i>"
        )

    def display(self):
        display(self.container)

## 5. Launch

Re-run this cell every time you change `VIDEO_PATH`.

In [16]:
annotator = Annotator(VIDEO_PATH, CSV_PATH)
annotator.display()

## Keyboard shortcuts

The annotator listens for keyboard events on the main interface container.

**Important:** You must click anywhere inside the annotator area (the status text, the image, or the buttons) to focus it before keyboard shortcuts will work.

**Using the Frame input:**
Simply click on the input box, type the number, and click **Go**. Then click back on the video image or status area to resume keyboard shortcuts.

| Key | Action |
|---|---|
| `Left` | back 1 frame |
| `Right` | forward 1 frame |
| `Shift + Left` | back 10 frames |
| `Shift + Right` | forward 10 frames |
| `s` or `Space` | label current frame as **straight** |
| `h` or `a` | label current frame as **hook** |
| `u` or `d` | label current frame as **uppercut** |
| `n` or `e` | label current frame as **none** (negative) |
| `o` | toggle "occluded" flag for the next label |
| `z` | undo last annotation for this video |
| `s` | save now |

## Workflow per video

1. Edit the config cell: set `VIDEO_PATH` to the video you want to annotate.
2. Run the launch cell.
3. Scrub right with arrow keys until a punch is imminent. Find the frame of **full arm extension**. Hit the class hotkey.
4. Continue to next punch. Auto-saves every 10 annotations; hit `s` to save manually.
5. When done with this video, change `VIDEO_PATH` and re-run the launch cell. The CSV accumulates.

## Consistency rule

Always label the same visual moment: **full arm extension** (the peak of the punch). Pick that definition on day one and don't drift.

## Performance note

PyAV seeks to the nearest keyframe, then decodes forward to the exact target. For codecs with sparse keyframes (e.g. every 5-10 seconds in some H.264 encodes), jumping backwards a few frames may take ~100-300ms as it re-decodes from the keyframe. Forward stepping is instant. If scrubbing feels sluggish on a specific file, re-encode with denser keyframes:

```
ffmpeg -i input.mp4 -c:v libx264 -g 10 -preset fast output.mp4
```

`-g 10` forces a keyframe every 10 frames, making backward seek near-instant.

## 6. Summary stats

Run any time to see progress.

In [8]:
# Quick sanity check on your annotations so far
df = pd.read_csv(CSV_PATH) if Path(CSV_PATH).exists() else pd.DataFrame()
if len(df):
    print(f"Total annotations: {len(df)}")
    print(f"Videos annotated:  {df['video_id'].nunique()}")
    print()
    print("Per class:")
    print(df['punch_type'].value_counts().to_string())
    print()
    print("Per video:")
    print(df.groupby('video_id').size().to_string())
    if 'occluded' in df.columns:
        n_occ = df['occluded'].sum() if df['occluded'].dtype != object else (df['occluded'] == True).sum()
        print(f"\nOccluded: {n_occ}")
else:
    print("No annotations yet.")

Total annotations: 2809
Videos annotated:  4

Per class:
punch_type
jab         1365
none         608
hook         480
uppercut     356

Per video:
video_id
V1.mp4     735
V5.mp4     616
V6.mp4    1189
V7.mp4     269

Occluded: 0
